# How the document chat works

This notebook walks through the same retrieval-augmented generation (RAG) pipeline as the Flask app in `web_app.py`. Both use the `rag_chatbot` package.

A question is answered only from PDFs that have been indexed. There is no built-in knowledge base and no paid API.

```
PDF text  ->  overlapping chunks  ->  embeddings  ->  ChromaDB
question  ->  embedding  ->  closest chunks  ->  local model  ->  answer + page excerpts
```

Run the cells from top to bottom. Ollama must be running (`ollama serve`) with `nomic-embed-text`. Answers use the local vLLM server. If vLLM is down, the answer cell falls back to `llama3.2:3b` on Ollama.

The notebook indexes a tiny file named `notebook-demo.pdf`, then deletes it in the last cell. Other uploads stay in the library.

## Setup

Settings such as model names and folders live in `rag_chatbot/config.py`. The next cell adds the project root to `sys.path`, so the imports work whether Jupyter was started from the project root or from another folder.

In [1]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "web_app.py").is_file():
    root = root.parent
if not (root / "web_app.py").is_file():
    raise SystemExit("Open this notebook from the ai-agent project.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from rag_chatbot import config
from rag_chatbot.ollama_client import chat_stream, embed_text, require_models
from rag_chatbot.uploads import (
    chunk_document_text,
    extract_pdf_pages,
    index_upload,
    prepare_document_answer,
    remove_upload,
    source_list,
)
from rag_chatbot.vectorstore import get_client, retrieve
from rag_chatbot.vllm_client import chat_stream_vllm

print("Project:", root)
print("Embeddings:", config.EMBED_MODEL, "at", config.OLLAMA_URL)
print("Answers:", config.VLLM_MODEL, "at", config.VLLM_URL)
print("Fallback chat model:", config.CHAT_MODEL)
print("Index folder:", config.UPLOAD_CHROMA_DIR)
print("Chunks kept per question:", config.TOP_K)

Project: /home/khushi/Desktop/repo/ai-agent
Embeddings: nomic-embed-text at http://127.0.0.1:11434
Answers: Qwen/Qwen2.5-7B-Instruct at http://127.0.0.1:8000/v1
Fallback chat model: llama3.2:3b
Index folder: /home/khushi/Desktop/repo/ai-agent/.chroma_uploads
Chunks kept per question: 6


## 1. Load and extract text from a PDF

A PDF is a page layout, not a plain text file. `extract_pdf_pages` uses `pypdf` to read the selectable text on each page and keeps the page number. Later answers cite that filename and page.

A scanned page is only a picture. This project does not run OCR, so a PDF with no text layer is rejected. The Flask upload route calls this same function.

The next cell builds a two-page sample in memory, writes it to a temporary file, and extracts the text. Page 2 holds the fact we will ask about.

In [2]:
import io

from pypdf import PdfReader, PdfWriter


def text_pdf(text: str) -> bytes:
    """Minimal one-page PDF. Real uploads are read the same way, with pypdf."""
    safe = text.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")
    stream = f"BT /F1 18 Tf 72 720 Td ({safe}) Tj ET".encode("latin-1")
    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R "
        b"/Resources << /Font << /F1 5 0 R >> >> >>",
        b"<< /Length %d >>\nstream\n" % len(stream) + stream + b"\nendstream",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
    ]
    out = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for i, obj in enumerate(objects, start=1):
        offsets.append(len(out))
        out.extend(f"{i} 0 obj\n".encode())
        out.extend(obj)
        out.extend(b"\nendobj\n")
    xref = len(out)
    out.extend(f"xref\n0 {len(objects)+1}\n".encode())
    out.extend(b"0000000000 65535 f \n")
    for offset in offsets[1:]:
        out.extend(f"{offset:010d} 00000 n \n".encode())
    out.extend(
        f"trailer << /Size {len(objects)+1} /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF".encode()
    )
    return bytes(out)


def two_page_pdf(first: str, second: str) -> bytes:
    writer = PdfWriter()
    for text in (first, second):
        reader = PdfReader(io.BytesIO(text_pdf(text)))
        writer.add_page(reader.pages[0])
    buffer = io.BytesIO()
    writer.write(buffer)
    return buffer.getvalue()


DEMO_NAME = "notebook-demo.pdf"
pdf_bytes = two_page_pdf(
    "This cover sheet only introduces the notebook demo.",
    "Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.",
)

sample_path = root / "data" / "uploads" / "_notebook_extract_probe.pdf"
sample_path.parent.mkdir(parents=True, exist_ok=True)
sample_path.write_bytes(pdf_bytes)
try:
    pages = extract_pdf_pages(sample_path)
finally:
    sample_path.unlink(missing_ok=True)

for page in pages:
    print(f"Page {page['page']}: {page['text']}")

Page 1: This cover sheet only introduces the notebook demo.
Page 2: Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.


## 2. Split text into overlapping chunks

A whole PDF can be too long to embed or to paste into one prompt. `chunk_document_text` splits text on blank lines. A paragraph longer than 700 characters is cut into windows of about 700 characters, with about 120 characters repeated. The cut prefers a space, so a sentence in the middle of a page is not sliced in half.

Each piece stays tied to the page it came from. The Flask app uses this same function while indexing an upload.

In [3]:
page_chunks = []
for page in pages:
    for piece in chunk_document_text(page["text"]):
        page_chunks.append({"page": page["page"], "text": piece})

print("Chunks from the sample PDF:")
for i, chunk in enumerate(page_chunks):
    print(f"{i}. page {chunk['page']}: {chunk['text']}")

long_text = ("alpha " * 116) + "The vault code is 9137." + (" omega" * 40)
long_chunks = chunk_document_text(long_text)
print()
print(f"A long paragraph becomes {len(long_chunks)} overlapping chunks.")
print("The vault code is still present:", any("9137" in piece for piece in long_chunks))
print("Second chunk starts with the tail of the first:", long_chunks[1][:40])

Chunks from the sample PDF:
0. page 1: This cover sheet only introduces the notebook demo.
1. page 2: Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.

A long paragraph becomes 2 overlapping chunks.
The vault code is still present: True
Second chunk starts with the tail of the first: ha alpha alpha alpha alpha alpha alpha a


## 3. Generate embeddings with the local model

An embedding is a long list of numbers that represents a piece of text. Strings with similar meaning land closer together than unrelated strings.

`embed_text` calls Ollama's `nomic-embed-text` model at `http://127.0.0.1:11434`. The Flask app uses that function for both PDF chunks and questions. A higher cosine similarity means the two strings are closer.

In [4]:
require_models(need_chat=False)

def cosine(left, right):
    dot = sum(x * y for x, y in zip(left, right))
    left_norm = sum(x * x for x in left) ** 0.5
    right_norm = sum(y * y for y in right) ** 0.5
    return dot / (left_norm * right_norm)

garden = embed_text("The garden gate code is 2468.")
related = embed_text("What is the code for the garden gate?")
unrelated = embed_text("How do I bake sourdough bread?")

print("Model:", config.EMBED_MODEL)
print("Embedding length:", len(garden))
print("First values:", [round(value, 4) for value in garden[:8]])
print("Related question cosine:", round(cosine(garden, related), 3))
print("Unrelated question cosine:", round(cosine(garden, unrelated), 3))

Model: nomic-embed-text
Embedding length: 768
First values: [0.1281, 0.0995, -3.715, 0.233, 1.1149, 0.1512, 0.3699, 0.7225]
Related question cosine: 0.869
Unrelated question cosine: 0.362


## 4. Store and retrieve chunks with ChromaDB

`index_upload` is the function behind `POST /api/upload`. It saves the PDF, embeds every chunk, and stores the text plus metadata in the local Chroma collection `user_uploads` under `.chroma_uploads/`.

Each chunk is saved with:

- `source`: the PDF filename
- `page`: the page number
- `chunk_id`: the position inside that file
- `sha256`: a fingerprint of the uploaded bytes

`retrieve` embeds the question and asks Chroma for the closest chunks. A smaller distance means a closer match. The app reports similarity as `1 - distance`. Only the selected filename is searched.

In [5]:
remove_upload(DEMO_NAME)
indexed = index_upload(DEMO_NAME, pdf_bytes)
print("Indexed:", indexed)

question = "What is the garden gate code?"
collection = get_client(config.UPLOAD_CHROMA_DIR).get_collection(config.UPLOAD_COLLECTION)
hits = retrieve(question, collection=collection, where={"source": DEMO_NAME})

print()
print("Closest chunks:")
for hit in hits:
    print(
        f"- {hit['source']}, page {hit['page']}  "
        f"distance={hit['distance']:.3f}  similarity={hit['similarity']:.3f}"
    )
    print(" ", hit["text"])

Indexed: {'name': 'notebook-demo.pdf', 'chunks': 2, 'pages': 2, 'duplicate': False, 'status': 'ready'}



Closest chunks:
- notebook-demo.pdf, page 2  distance=0.185  similarity=0.815
  Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.
- notebook-demo.pdf, page 1  distance=0.592  similarity=0.408
  This cover sheet only introduces the notebook demo.


## 5. Generate an answer with vLLM, or Ollama if vLLM is down

`prepare_document_answer` is what `POST /api/chat` calls before the model. It keeps only the selected PDFs, builds a prompt from the closest excerpts, and tells the model to cite the filename and page. If the excerpts do not contain the fact, the model is told to say so.

The first answer attempt is the local vLLM server (`Qwen/Qwen2.5-7B-Instruct` at `http://127.0.0.1:8000/v1`). If that request fails, the same messages are sent to Ollama's `llama3.2:3b`. Embeddings stay on Ollama either way.

In [6]:
prepared = prepare_document_answer(question, [DEMO_NAME])
print(prepared["messages"][-1]["content"])
print()

def stream_answer(messages):
    """Same order as web_app.py: vLLM first, Ollama chat if vLLM fails."""
    try:
        pieces = list(chat_stream_vllm(messages))
        print("Answered by vLLM:", config.VLLM_MODEL)
        return "".join(pieces)
    except Exception as exc:
        print(f"vLLM unavailable ({exc}). Falling back to Ollama {config.CHAT_MODEL}.")
        return "".join(chat_stream(messages))

answer = stream_answer(prepared["messages"])

CONTEXT:
[1] (notebook-demo.pdf p.2)
Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.

[2] (notebook-demo.pdf p.1)
This cover sheet only introduces the notebook demo.

QUESTION:
What is the garden gate code?



Answered by vLLM: Qwen/Qwen2.5-7B-Instruct


## 6. Show the answer with page references and evidence excerpts

`source_list` is the same list the Flask page renders under the answer. Each item has the filename, the page number, and the excerpt that was retrieved. The excerpt is the evidence for the reply, not a second model call.

In [7]:
excerpts = source_list(prepared["hits"])

print("Answer")
print("------")
print(answer)
print()
print("Evidence")
print("--------")
for item in excerpts:
    where = f"{item['name']}, page {item['page']}" if item.get("page") else item["name"]
    print(where)
    print(item["text"])
    print()

Answer
------
The garden gate code is 2468. This information is cited from notebook-demo.pdf on page 2.

Evidence
--------
notebook-demo.pdf, page 2
Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.

notebook-demo.pdf, page 1
This cover sheet only introduces the notebook demo.



## Clean up the lesson file

Run this cell when you are finished. It deletes `notebook-demo.pdf` and its chunks. Other uploaded PDFs are left alone.

In [8]:
remove_upload(DEMO_NAME)
print("Removed", DEMO_NAME)

Removed notebook-demo.pdf
